# Lesson 07 Lab — Inference Precision Layers: Weights, Activations, and KV Cache

**Puzzle:** When a model is called INT4, which tensors are actually four-bit?

This notebook keeps the RTX 5090 outputs from a complete run. Read the theory cells, make a prediction, and then use **Run All** on your own GPU.


## Why this matters

Calling a model INT4 usually describes only part of its state. Weight-only layers may store four-bit codes while activations and accumulators use BF16, the KV cache grows with context, and temporary workspaces appear only at runtime. Capacity planning fails when those objects are collapsed into one advertised precision.


## 0. Predict before running

1. Write the KV-cache byte formula before looking at the projected values.
2. Predict which memory account grows with sequence length and which stays fixed for a loaded model.
3. Explain why checkpoint size cannot predict peak CUDA allocation by itself.

For each answer, name the observation that would prove you wrong.


## 1. Name the concrete objects

Inference precision belongs to separate ledgers: persistent weights, per-step activations/workspaces, accumulators, and persistent-per-request KV cache. Weight-only INT4 normally leaves activation and accumulation formats wider.

- Weight-only quantization leaves activations and accumulation in a floating-point compute dtype.
- KV cache grows with layers, sequence length, key/value heads, head dimension, batch, and cache dtype.
- Peak memory also includes temporary workspaces and allocator reserve.


## 2. Derive the mechanism

For a standard cache, `bytes = 2 × layers × batch × sequence × kv_heads × head_dim × bytes_per_element`; the leading two is for keys and values. Grouped-query attention changes `kv_heads`, not the number of query heads.

For a decoder cache with batch B, layers L, sequence S, KV heads H, head dimension D, two tensors K and V, and b bytes per element, the leading storage is `2·B·L·S·H·D·b`. Weight storage is roughly `parameters × effective bits/8` plus scales and unquantized tensors. Activations depend on execution phase and liveness, while workspaces and allocator reserve depend on backend behavior.

These terms have different lifetimes. Weights persist after load, KV cache persists per active request, and many activations are temporary. That makes concurrency a multiplication on the cache term, not on the model weights. The ledger must keep bytes, lifecycle, and ownership together.

### Mechanism at a glance

```mermaid
flowchart LR
  W["Weights<br/>persistent"] --> K["Layer kernel"]
  A["Activations<br/>short-lived"] --> K
  C["KV cache<br/>grows with context"] <--> K
  K --> O["Output activations"]
  W -. "storage dtype may differ<br/>from compute dtype" .-> K
```

### Walk it step by step

1. **Separate persistent from temporary state.** Weights persist for the model lifetime; activations and workspace live for an operator or layer.
2. **Account for context state.** KV cache grows with layers, batch, sequence length, heads, and head dimension.
3. **Name storage and compute dtypes.** A tensor stored in INT4 may be dequantized into FP16/BF16 before or inside the kernel.
4. **Optimize the dominant term.** Choose weight, activation, or KV quantization only after the workload-specific memory ledger identifies the bottleneck.


## 3. Verify the execution environment

The next cell asserts CUDA availability, fixes the seed, locates the lesson, and prints a sanitized GPU/PyTorch/CUDA record. Check it before interpreting output.


In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "07-inference-precision-layers"
device = require_cuda()
torch.manual_seed(2026 + 7)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 4. Freeze the comparison

| Role | This run |
|---|---|
| Baseline | BF16 KV-cache projection and a real BF16 K/V allocation |
| Candidate | INT8 cache projection for the same model geometry |
| Held constant | batch 1, 32 layers, 8 KV heads, head dimension 128, identical context lengths |
| Measurements | projected cache GiB by context and byte count of an allocated representative tensor pair |
| Evidence | `pytorch-gpu` |

**Experiment:** Build a memory ledger and allocate representative BF16 and INT8 KV tensors on CUDA to validate element-count arithmetic.


## 5. Read the experiment code

The lab validates the KV element-count formula with a live allocation and projects several context lengths without pretending to allocate a full model.

The notebook first calculates the formula for three sequence lengths, then allocates representative K and V tensors on CUDA and checks their exact element-count bytes. This joins arithmetic with a live tensor object without pretending to load a full model.

Scales, paging fragmentation, prefix-cache blocks, and temporary attention workspaces are intentionally outside the simple projection. They belong in the next ledger revision when a named serving backend is tested.

Only after these variables match the protocol should the cell be executed.


In [2]:
cfg = {"layers": 32, "kv_heads": 8, "head_dim": 128, "batch": 1}
rows = []
for seq in (2048, 8192, 32768):
    elements = 2 * cfg["layers"] * cfg["batch"] * seq * cfg["kv_heads"] * cfg["head_dim"]
    rows.append({"sequence": seq, "bf16_gib": round(elements*2/2**30, 4), "int8_gib": round(elements/2**30, 4)})
k = torch.empty(2, 4096, 8, 128, device=device, dtype=torch.bfloat16)
actual = k.numel() * k.element_size()
result = base_result(7, "pytorch-gpu"); result.update({"configuration": cfg, "projected_kv": rows,
    "allocation_probe": {"shape": list(k.shape), "bytes": actual, "dtype": str(k.dtype)},
    "conclusion": "Weights, activations, and KV cache require separate precision and memory ledger entries."})


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| BF16 KV at 2,048 tokens | 0.250 GiB |
| BF16 KV at 8,192 tokens | 1.000 GiB |
| BF16 KV at 32,768 tokens | 4.000 GiB |
| INT8 KV at 32,768 tokens | 2.000 GiB |
| Live allocation probe | 16,777,216 bytes |


## 7. Interpret rather than merely print

For the fixed 32-layer geometry, projected BF16 KV storage was 0.25 GiB at 2,048 tokens, 1.0 GiB at 8,192, and 4.0 GiB at 32,768. The INT8 arithmetic projection was exactly half each value. The live probe allocated two BF16 tensors of shape `[2, 4096, 8, 128]` totaling 16,777,216 bytes.

The linear fourfold growth from 8K to 32K is the important systems result. Weight quantization does not change it. Cache quantization may increase feasible context or concurrency, but only after scale overhead, attention compatibility, error, and latency are measured.

**Inspection rule:** Report each object separately. A checkpoint-size reduction does not establish the same reduction in runtime peak memory.


## 8. Keep the evidence label honest

This run is labeled **`pytorch-gpu`**. The measured tensors and operations ran on CUDA through PyTorch. The result does not name a separate production backend unless an operator trace identifies it.

The next cell writes the complete structured result; its existing saved output is part of the checked-in evidence.


In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "allocation_probe": {
    "bytes": 16777216,
    "dtype": "torch.bfloat16",
    "shape": [
      2,
      4096,
      8,
      128
    ]
  },
  "conclusion": "Weights, activations, and KV cache require separate precision and memory ledger entries.",
  "configuration": {
    "batch": 1,
    "head_dim": 128,
    "kv_heads": 8,
    "layers": 32
  },
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "pytorch-gpu",
  "executed_at_utc": "2026-08-07T14:45:23+00:00",
  "lesson": 7,
  "projected_kv": [
    {
      "bf16_gib": 0.25,
      "int8_gib": 0.125,
      "sequence": 2048
    },
    {
      "bf16_gib": 1.0,
      "int8_gib": 0.5,
      "sequence": 8192
    },
    {
      "bf16_gib": 4.0,
      "int8_gib": 2.0,
      "sequence": 32768
    }
  ],
  "schema_version": 1
}
Saved: artifacts/rtx5090-result.json


## 9. Make the bounded decision

> Name the object and lifecycle whenever you name a precision: weights, activations, accumulators, or cache.

**Acceptance/rollback:** Measure allocated/reserved/peak memory separately and reconcile them with object-level arithmetic. A checkpoint byte count is not a runtime memory result.

**Failure analysis:** A common error is multiplying weight memory by request count or forgetting to multiply cache by layers and by both K and V. Another is treating free memory reported before model load as deployable capacity. Allocator reserve, CUDA graphs, kernels, and safety margin must be added before setting concurrency.


## 10. Extend the evidence

Extend the ledger with grouped-query attention variants, tensor parallel sharding, cache block size, scale metadata, and allocator fragmentation. Then run a vLLM or TensorRT-LLM server and compare predicted versus observed cache capacity at 2K, 8K, and 32K contexts.

The full derivation, reproduction command, evidence boundary and primary references are in [`README.md`](README.md).
